# Evaluation
Evaluate a trained model and save confusion matrix to outputs.

In [3]:
from pathlib import Path
import sys
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import METADATA_CSV
from src.data.metadata import load_metadata, filter_by_set, material_label_map
from src.data.dataset import TWITorchDataset
from src.data.loader import load_mat_signal
from src.models.cnn1d import CNN1D
from src.evaluation.metrics import accuracy, macro_precision_recall_f1
from src.evaluation.confusion_matrix import confusion_matrix, plot_confusion_matrix
from src.processing.preprocess import preprocess_signal
from src.processing.music import music_spectrum
from src.visualization.plot_music import plot_music_spectrum

base_folder = ROOT / "data" / "36"
mat_key = "dataMeasured1"

torch.manual_seed(42)

metadata = load_metadata(METADATA_CSV)
metadata = filter_by_set(metadata, "single")
metadata = metadata[metadata["material"].notna()]
label_map = material_label_map(metadata)

file_paths = []
labels = []
for _, row in metadata.iterrows():
    mat_path = base_folder / str(int(row["folder_id"])) / "data.mat"
    if mat_path.exists():
        file_paths.append(mat_path)
        labels.append(label_map[row["material"]])

dataset = TWITorchDataset(file_paths, labels, mat_key=mat_key, transform=preprocess_signal)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_set, test_set = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8, shuffle=False)

sample_x, _ = dataset[0]
model = CNN1D(in_channels=int(sample_x.shape[0]), num_classes=len(label_map))

train_labels = [labels[i] for i in train_set.indices]
class_counts = np.bincount(train_labels, minlength=len(label_map))
class_weights = 1.0 / np.maximum(class_counts, 1)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for _ in range(30):
    for x, y in train_loader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

out_models = ROOT / "outputs" / "models"
out_models.mkdir(parents=True, exist_ok=True)
model_path = out_models / "best_model_nb.pt"
torch.save(model.state_dict(), model_path)

model.eval()
y_true = []
y_pred = []
with torch.no_grad():
    for x, y in test_loader:
        logits = model(x)
        preds = torch.argmax(logits, dim=1).tolist()
        y_true.extend(y.tolist())
        y_pred.extend(preds)

acc = accuracy(y_true, y_pred)
prf = macro_precision_recall_f1(y_true, y_pred)
print("Accuracy:", acc)
print("Precision/Recall/F1:", prf)

label_names = {v: k for k, v in label_map.items()}
cm_labels, cm = confusion_matrix(y_true, y_pred)
cm_display = [label_names[i] for i in cm_labels]

out_fig = ROOT / "outputs" / "figures"
out_fig.mkdir(parents=True, exist_ok=True)
cm_path = out_fig / "confusion_matrix_nb.png"
plot_confusion_matrix(cm_display, cm, cm_path)
print("Saved confusion matrix to", cm_path)

sample_signal = load_mat_signal(file_paths[0], key=mat_key)
processed = preprocess_signal(sample_signal)
spectrum = music_spectrum(processed, num_sources=1, n_fft=256)
peak_idx = int(np.argmax(spectrum))

music_path = out_fig / "hidden_target_music.png"
plot_music_spectrum(spectrum, music_path)
print("MUSIC peak bin:", peak_idx)
print("Saved MUSIC spectrum to", music_path)

Accuracy: 0.5
Precision/Recall/F1: {'precision': 0.25, 'recall': 0.5, 'f1': 0.3333333333333333}
Saved confusion matrix to /Users/abhishekgautam/Desktop/TWI-project/outputs/figures/confusion_matrix_nb.png
MUSIC peak bin: 253
Saved MUSIC spectrum to /Users/abhishekgautam/Desktop/TWI-project/outputs/figures/hidden_target_music.png
